# Phase 2 — sweep the pipeline on one model

Everything that is **not** the model is compared here on `baseline_cnn`, so each comparison
is controlled and cheap. Only a setting that wins gets carried to the real architectures.

**The sweep runs from `scripts/sweep.py`, not from this notebook.** A 2.5-hour run outlives
a Colab kernel; a terminal process survives a disconnect, a notebook cell does not. This
notebook sets up, launches, and reads results.

| stage | arms | runs |
|---|---|---|
| 0 | control — defaults untouched | 1 |
| 1 | `dihedral8` · `rotation` · `dihedral8_rotation` | 3 |
| 2 | `unweighted_ce` · `focal_loss` | 2 |
| 3 | `cbam` | 1 |
| 4 | `grayscale_rgb` | 1 |
| 5 | 224² | 1 |
| 6 | `cosine` — conditional | 1 |
| 7 | post-hoc: TTA, thresholds | 0 |

**Run stage 0 alone first.** Two later decisions depend on it: where early stopping fires
decides whether stage 6 is worth running at all (a cosine annealed over 50 epochs barely
acts if the run stops at 20), and the per-class table says whether `Scratch` is weak enough
to make the 224² arm a priority rather than a curiosity.

Each stage is meant to start from the previous stage's winner. That hand-off is manual on
purpose: picking a winner among overlapping confidence intervals is a judgement, not an
argmax.

In [ ]:
1

In [ ]:
import wandb, pandas as pd
runs = wandb.Api().runs("vlad-yelisieiev-bicocca-milano-bicocca/wm811k-wafer-defects")
df = pd.DataFrame([{"run": r.name, "macro_f1": r.summary.get("best_metric"),
                    "best_epoch": r.summary.get("best_epoch"),
                    "epochs": r.summary.get("epoch")} for r in runs])
print(df.sort_values("macro_f1", ascending=False))

## 1. Setup

Same bootstrap as phase 0; every step is a no-op if already done.

In [21]:
import os, subprocess, sys
from pathlib import Path


def looks_like_the_repository(path: Path) -> bool:
    return (path / "pyproject.toml").exists() and (path / "src" / "fdl_project").is_dir()


REPO = next(
    (p for p in [Path.cwd(), *Path.cwd().parents, Path("/content/fdl-project")]
     if looks_like_the_repository(p)),
    None,
)
assert REPO is not None, "clone the repo to /content/fdl-project first"
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--ignore-requires-python",
                "-e", str(REPO), "--no-deps"], check=True)
source = str(REPO / "src")
if source not in sys.path:
    sys.path.insert(0, source)

import torch
print(f"repo {REPO}")
print(f"gpu  {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

repo /content/fdl-project
gpu  Tesla T4


In [22]:
import shutil
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")
DRIVE = DRIVE_ROOT / "BICOCCA/FDL"
DATASET = REPO / "data/MIR-WM811K/WM811K.pkl"
EXPECTED_BYTES = 2_022_961_642


def mount_drive() -> bool:
    """Mount if needed. Idempotent: never re-prompts, never drops a live mount."""

    if DRIVE_ROOT.is_dir():
        return True
    try:
        from google.colab import drive

        drive.mount("/content/drive")     # no force_remount: that tears down a mount
    except Exception as error:
        print(f"  Drive unavailable ({type(error).__name__})")
        return False
    return DRIVE_ROOT.is_dir()


HAS_DRIVE = mount_drive()

# Every fresh VM starts empty, so this runs each session. Copy to /content and
# train from there: Drive is a FUSE network mount, fine for one sequential read
# and poor for the repeated access a training loop makes.
if not DATASET.exists():
    DATASET.parent.mkdir(parents=True, exist_ok=True)
    source = DRIVE / "DATA/data/MIR-WM811K/WM811K.pkl"
    if HAS_DRIVE and source.exists():
        print(f"copying {source.name} from Drive ...")
        shutil.copy2(source, DATASET)
    else:
        import tempfile, urllib.request, zipfile

        url = "http://mirlab.org/dataset/public/MIR-WM811K.zip"
        print(f"Drive copy not found; downloading (~328 MB) from {url}")
        with tempfile.TemporaryDirectory() as work:
            archive = Path(work) / "wm811k.zip"
            urllib.request.urlretrieve(url, archive)
            with zipfile.ZipFile(archive) as bundle:   # skip the 3.6 GB .mat
                member = next(n for n in bundle.namelist() if n.endswith("WM811K.pkl"))
                with bundle.open(member) as src, open(DATASET, "wb") as dst:
                    shutil.copyfileobj(src, dst)

size = DATASET.stat().st_size
assert size == EXPECTED_BYTES, (
    f"got {size:,} bytes, want {EXPECTED_BYTES:,}. The splits are row indices "
    "into the original pickle, so a different upload silently selects "
    "different wafers and breaks the lot-grouping."
)
print(f"drive   {'mounted' if HAS_DRIVE else 'NOT mounted'}")
print(f"dataset {size / 1024**3:.2f} GiB at {DATASET}")

drive   mounted
dataset 1.88 GiB at /content/fdl-project/data/MIR-WM811K/WM811K.pkl


: 

: 

## 2. Checkpoints on Drive

**Do this before launching.** Local Colab disk is wiped when the session drops, and the
sweep is long enough that it will. `resume: auto` restores optimizer, schedule, scaler,
epoch, best-so-far, history and RNG state — but only if the files outlive the session.

Training still reads the *dataset* from `/content`: Drive is a FUSE network mount, fine for
checkpoint writes and poor for the repeated reads a training loop makes.

In [23]:
CHECKPOINTS = DRIVE / "checkpoints"
if HAS_DRIVE:
    CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    print(f"checkpoints -> {CHECKPOINTS}")
else:
    print("Drive not mounted: checkpoints stay on local disk and die with the session")

checkpoints -> /content/drive/MyDrive/BICOCCA/FDL/checkpoints


: 

## 3. Weights & Biases

Optional, and off unless a config asks for it.

**Colab Secrets do not work here.** `userdata.get()` talks to the Colab web frontend, which
does not exist when the runtime is driven from VS Code -- it times out with *"Secrets can
only be fetched when running from the Colab UI"* no matter what is set on the web UI.

**Run `wandb login` in the terminal instead**, once per session:

```bash
wandb login          # paste the key from wandb.ai/authorize
```

That writes `~/.netrc` on the VM, which both the terminal and this kernel read. An
environment variable would only reach the process that set it -- and the sweep runs from a
terminal, which inherits nothing from this notebook.

`~/.netrc` lives on the VM, so it is gone on the next fresh runtime, same as the dataset.

The cell below still tries Secrets and the environment first, so it works unchanged if you
ever run this from the Colab web UI. Leave `entity` as `null` and runs land in your own
account; set it to a team name once one exists.

In [24]:
import os, subprocess, sys

USE_WANDB = True
WANDB_PROJECT = "wm811k-wafer-defects"
WANDB_ENTITY = None       # None = your own account; a team name once one exists

if USE_WANDB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)
    import wandb

    # `wandb login` in the terminal writes ~/.netrc, which the library reads on
    # its own -- so if that was done, there is nothing to do here.
    authenticated = bool(wandb.api.api_key)

    if not authenticated:
        # Fall back to a key in the environment, then Colab Secrets. Secrets
        # only answer from the Colab web UI; from VS Code the call times out.
        key = os.environ.get("WANDB_KEY") or os.environ.get("WANDB_API_KEY")
        if key is None:
            try:
                from google.colab import userdata

                key = userdata.get("WANDB_KEY")
            except Exception as error:
                print(f"  Colab Secrets unavailable ({type(error).__name__})")
        if key:
            os.environ["WANDB_API_KEY"] = key
            del key
            authenticated = True

    if authenticated:
        print(f"  wandb authenticated, project {WANDB_PROJECT!r}")
    else:
        USE_WANDB = False
        print("  not authenticated -- run `wandb login` in the terminal, then rerun")

  wandb authenticated, project 'wm811k-wafer-defects'


In [25]:
# Passed to every arm. Per-machine settings belong here, not in the configs --
# those describe experiments, not environments.
SWEEP_OVERRIDES = []
if HAS_DRIVE:
    SWEEP_OVERRIDES.append(f"checkpoint.directory={CHECKPOINTS}")
if USE_WANDB:
    SWEEP_OVERRIDES += [
        "logging.wandb.enabled=true",
        f"logging.wandb.project={WANDB_PROJECT}",
        *([f"logging.wandb.entity={WANDB_ENTITY}"] if WANDB_ENTITY else []),
    ]
print("\n".join(f"  --override {o}" for o in SWEEP_OVERRIDES) or "  (none)")

  --override checkpoint.directory=/content/drive/MyDrive/BICOCCA/FDL/checkpoints
  --override logging.wandb.enabled=true
  --override logging.wandb.project=wm811k-wafer-defects


## 4. Launch from the terminal

`Cmd+Shift+P` → **Colab: Open Terminal**, then:

```bash
cd /content/fdl-project

# stage 0 first, on its own
python scripts/sweep.py --stage 0 \
    --override checkpoint.directory=/content/drive/MyDrive/BICOCCA/FDL/checkpoints

# then, once you have read the control's curve
nohup python scripts/sweep.py --stage 1 2 3 4 5 > sweep.log 2>&1 &
tail -f sweep.log
```

`nohup ... &` detaches it, so closing the terminal or losing the notebook kernel does not
kill the sweep.

**It is resumable.** Results are appended to `output/phase2/results.csv` after each arm, and
re-running skips anything already recorded — so after a disconnect you re-run the same
command and it picks up where it stopped. A failed arm is recorded as failed and the sweep
continues rather than dying.

`--seeds 86 87 88` runs each arm three times. At ~6 min per 64² run that is the difference
between "these twelve numbers overlap" and knowing which differences are real.

In [26]:
# Copy this into the Colab terminal. Running it there rather than here is the
# point: a detached process survives the kernel dropping, a cell does not.
flags = " ".join(f"--override {o}" for o in SWEEP_OVERRIDES)
print("cd /content/fdl-project\n")
print(f"python scripts/sweep.py --stage 0 {flags}\n")
print(f"nohup python scripts/sweep.py --stage 1 2 3 4 5 {flags} > sweep.log 2>&1 &")
print("tail -f sweep.log")

cd /content/fdl-project

python scripts/sweep.py --stage 0 --override checkpoint.directory=/content/drive/MyDrive/BICOCCA/FDL/checkpoints --override logging.wandb.enabled=true --override logging.wandb.project=wm811k-wafer-defects

nohup python scripts/sweep.py --stage 1 2 3 4 5 --override checkpoint.directory=/content/drive/MyDrive/BICOCCA/FDL/checkpoints --override logging.wandb.enabled=true --override logging.wandb.project=wm811k-wafer-defects > sweep.log 2>&1 &
tail -f sweep.log


In [27]:
# Or launch it from here — same process, but it dies with the kernel.
# Prefer the terminal for anything longer than a single arm.
!python scripts/sweep.py --list

  stage 0  baseline_cnn-baseline               defaults untouched
  stage 1  baseline_cnn-dihedral8              all 8 square symmetries, exact permutations
  stage 1  baseline_cnn-rotation               free angle, resamples, so half the samples
  stage 1  baseline_cnn-dihedral8-rotation     group always, rotation on half
  stage 2  baseline_cnn-unweighted-ce          the baseline the sampler's gain is measured against
  stage 2  baseline_cnn-focal-loss             compare against unweighted, not stacked on the sampler
  stage 3  baseline_cnn-cbam                   +610 parameters
  stage 4  baseline_cnn-grayscale              does the false ordering hurt?
  stage 5  baseline_cnn-224px                  224 is near-native (max 212x187); 64 is the compression
  stage 6  baseline_cnn-cosine                 only meaningful if the control runs near max_epochs


## 5. Results

Re-run this cell any time; it reads the CSV the sweep appends to.

In [28]:
import pandas as pd

RESULTS = REPO / "output/phase2/results.csv"
if not RESULTS.is_file():
    print("no results yet — start with stage 0")
else:
    results = pd.read_csv(RESULTS)
    control = results.query("arm == 'control'")["macro_f1"].mean()

    view = results.copy()
    if pd.notna(control):
        view["vs control"] = (view["macro_f1"] - control).round(4)
        # An arm only counts as different if its interval clears the control's
        # point estimate; with nine classes and 15 test images in the rarest,
        # point estimates alone are not a ranking.
        view["clears control"] = view["ci_lower"] > control
    display(view.sort_values(["stage", "macro_f1"], ascending=[True, False]))

,run,stage,arm,seed,macro_f1,ci_lower,ci_upper,best_epoch,epochs_run,minutes,overrides,note
0,baseline_cnn-baseline-s86,0,baseline,86,0.8152,0.7993,0.8297,7.0,12.0,2.8,NaN,defaults untouched
1,baseline_cnn-dihedral8-s86,1,dihedral8,86,0.8582,0.8428,0.8703,15.0,20.0,4.8,data.augmentation.name=dihedral8,"all 8 square symmetries, exact permutations"
2,baseline_cnn-rotation-s86,1,rotation,86,NaN,NaN,NaN,NaN,NaN,0.2,data.augmentation.name=rotation data.augmentat...,FAILED: Caught ForkedError in DataLoader worke...
3,baseline_cnn-dihedral8-rotation-s86,1,dihedral8-rotation,86,NaN,NaN,NaN,NaN,NaN,0.2,data.augmentation.name=dihedral8_rotation,FAILED: Caught ForkedError in DataLoader worke...
4,baseline_cnn-unweighted-ce-s86,2,unweighted-ce,86,0.8206,0.8062,0.8337,15.0,20.0,4.2,imbalance.preset=unweighted_ce,the baseline the sampler's gain is measured ag...
5,baseline_cnn-focal-loss-s86,2,focal-loss,86,0.8204,0.8034,0.8351,23.0,28.0,5.8,imbalance.preset=focal_loss,"compare against unweighted, not stacked on the..."
6,baseline_cnn-cbam-s86,3,cbam,86,0.8127,0.7967,0.8276,5.0,10.0,2.3,model.kwargs.attention=cbam,+610 parameters
7,baseline_cnn-grayscale-s86,4,grayscale,86,0.8025,0.7869,0.8164,5.0,10.0,1.9,data.preprocessing.encoding=grayscale_rgb,does the false ordering hurt?
8,baseline_cnn-224px-s86,5,224px,86,0.8321,0.8170,0.8469,14.0,19.0,24.4,"data.preprocessing.target_size=[224,224]",224 is near-native (max 212x187); 64 is the co...


In [29]:
# Seed spread, when more than one seed has been run. This is what says whether
# a difference between two arms is real or noise.
if RESULTS.is_file() and results["seed"].nunique() > 1:
    spread = (results.groupby("arm")["macro_f1"]
              .agg(["mean", "std", "count"]).round(4)
              .sort_values("mean", ascending=False))
    display(spread)
    print(f"\ntypical seed-to-seed std: {results.groupby('arm')['macro_f1'].std().median():.4f}")
    print("A gap between arms smaller than that is not a result.")
else:
    print("single seed — run --seeds 86 87 88 before ranking anything")

single seed — run --seeds 86 87 88 before ranking anything


## 6. After the sweep

1. **Write the winners into `docs/experiment-grid.md`** — the cells are there to be filled.
   Record arms that were indistinguishable too: "five imbalance policies were within noise
   at 50 epochs" is a result.
2. **Save the settled pipeline as a config** (`configs/train/pipeline.yaml`) so phase 3
   inherits it by reference rather than by ten copied override flags.
3. **Post-hoc last** — TTA and per-class thresholds need no retraining:
   ```bash
   python scripts/inference.py --checkpoint output/runs/<winner>/best_model.pt \
       --split validation --tta --tune-thresholds
   ```

**One interaction to check by hand.** The sampler and augmentation compound — a rare wafer
drawn 40× an epoch already yields 40 distinct views. If augmentation wins stage 1, re-run
the best stage-2 loss arm with augmentation both on and off before adopting either. That is
the one place the greedy stage order can mislead you.